# RAG Framework - Interactive Debug Notebook
**Purpose:** Visualize every step of the RAG pipeline with debug logs

**Structure:**
- **Part 1: INDEXING** (Retrieval Setup)
  - PDF Text Extraction
  - Text Chunking
  - Embedding Generation
  - Vector Storage
- **Part 2: QUERYING** (Augmentation + Generation)
  - Question Embedding
  - Semantic Search (Retrieval)
  - Context Building (Augmentation)
  - LLM Answer Generation

**Goal:** See EXACTLY what each component does with real data

---
# Setup & Imports

In [1]:
# Install required packages (run only once)
# !pip install PyPDF2 langchain chromadb sentence-transformers together pandas numpy

In [2]:
import PyPDF2
from langchain.text_splitter import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb
import numpy as np
import pandas as pd
from pprint import pprint
import warnings
warnings.filterwarnings('ignore')

print("✅ All packages imported successfully!")

/Users/I772947/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/I772947/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ All packages imported successfully!


In [3]:
# Configuration
CHUNK_SIZE = 1500
CHUNK_OVERLAP = 100
TOP_K = 5  # Number of chunks to retrieve
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

print(f"📋 Configuration:")
print(f"   CHUNK_SIZE: {CHUNK_SIZE} characters")
print(f"   CHUNK_OVERLAP: {CHUNK_OVERLAP} characters")
print(f"   TOP_K: {TOP_K} chunks")
print(f"   EMBEDDING_MODEL: {EMBEDDING_MODEL}")

📋 Configuration:
   CHUNK_SIZE: 1500 characters
   CHUNK_OVERLAP: 100 characters
   TOP_K: 5 chunks
   EMBEDDING_MODEL: all-MiniLM-L6-v2


---
---
# PART 1: INDEXING (RETRIEVAL SETUP)
**This happens ONCE per document before asking questions**

## Step 1.1: PDF Text Extraction
**Goal:** Extract raw text from PDF file

**What you'll see:**
- Number of pages in PDF
- Raw text from first 2 pages
- Total characters extracted

In [4]:
# Select a PDF from your Sprint 3 dataset
# Change this path to any PDF you want to test
PDF_PATH = "Sprint 3/UDA-Benchmark/dataset/src_doc_files_example/fin/ADI_2009.pdf"

print(f"🔍 Loading PDF: {PDF_PATH}")
print("="*80)

🔍 Loading PDF: Sprint 3/UDA-Benchmark/dataset/src_doc_files_example/fin/ADI_2009.pdf


In [5]:
# Extract text from PDF
def extract_text_from_pdf(pdf_path):
    """Extract text from PDF and return full text + page-by-page breakdown"""
    with open(pdf_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)
        num_pages = len(pdf_reader.pages)
        
        print(f"📄 PDF Info:")
        print(f"   Total Pages: {num_pages}")
        
        # Extract text from all pages
        pages_text = []
        full_text = ""
        
        for i, page in enumerate(pdf_reader.pages):
            page_text = page.extract_text()
            pages_text.append(page_text)
            full_text += page_text + "\n"
            
        print(f"   Total Characters: {len(full_text):,}")
        print(f"   Total Words (approx): {len(full_text.split()):,}")
        
        return full_text, pages_text, num_pages

# Extract
full_text, pages_text, num_pages = extract_text_from_pdf(PDF_PATH)
print("\n✅ Text extraction complete!")

FileNotFoundError: [Errno 2] No such file or directory: 'Sprint 3/UDA-Benchmark/dataset/src_doc_files_example/fin/ADI_2009.pdf'

In [ ]:
# 🔍 DEBUG: Show raw text from first 2 pages
print("="*80)
print("🔍 DEBUG: Raw Text from Page 1")
print("="*80)
print(pages_text[0][:1000])  # First 1000 characters
print("\n... (truncated) ...\n")

print("="*80)
print("🔍 DEBUG: Raw Text from Page 2")
print("="*80)
print(pages_text[1][:1000])  # First 1000 characters
print("\n... (truncated) ...\n")

## Step 1.2: Text Chunking
**Goal:** Split large text into smaller overlapping chunks

**What you'll see:**
- Number of chunks created
- Size of each chunk
- First 3 chunks (full text)
- Overlap between chunks

In [ ]:
# Split text into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""]  # Try to split at natural boundaries
)

chunks = text_splitter.split_text(full_text)

print(f"📊 Chunking Results:")
print(f"   Total Chunks Created: {len(chunks)}")
print(f"   Chunk Size: {CHUNK_SIZE} characters")
print(f"   Chunk Overlap: {CHUNK_OVERLAP} characters")
print(f"   Average Chunk Size: {np.mean([len(c) for c in chunks]):.0f} characters")
print(f"   Min Chunk Size: {min(len(c) for c in chunks)} characters")
print(f"   Max Chunk Size: {max(len(c) for c in chunks)} characters")
print("\n✅ Chunking complete!")

In [ ]:
# 🔍 DEBUG: Show first 3 chunks
print("="*80)
print("🔍 DEBUG: First 3 Chunks (Full Text)")
print("="*80)

for i in range(min(3, len(chunks))):
    print(f"\n{'='*80}")
    print(f"CHUNK {i+1} (Length: {len(chunks[i])} chars)")
    print(f"{'='*80}")
    print(chunks[i])
    print(f"\n[End of Chunk {i+1}]")

In [ ]:
# 🔍 DEBUG: Show overlap between chunks
print("="*80)
print("🔍 DEBUG: Overlap Between Chunks")
print("="*80)

if len(chunks) >= 2:
    # Find overlap between chunk 1 and chunk 2
    chunk1_end = chunks[0][-200:]  # Last 200 chars of chunk 1
    chunk2_start = chunks[1][:200]  # First 200 chars of chunk 2
    
    print("\n📍 Last 200 chars of Chunk 1:")
    print(chunk1_end)
    print("\n📍 First 200 chars of Chunk 2:")
    print(chunk2_start)
    
    # Check for overlap
    overlap_found = False
    for i in range(len(chunk1_end)):
        if chunk1_end[i:] == chunk2_start[:len(chunk1_end)-i]:
            overlap_text = chunk1_end[i:]
            if len(overlap_text) > 10:
                print(f"\n✅ Found overlap of {len(overlap_text)} characters:")
                print(f"\n>>> {overlap_text} <<<")
                overlap_found = True
                break
    
    if not overlap_found:
        print("\n⚠️ No clear overlap detected (chunks may be at natural boundaries)")

## Step 1.3: Embedding Generation
**Goal:** Convert text chunks into numerical vectors (embeddings)

**What you'll see:**
- Embedding model details
- Shape of embeddings (number × dimensions)
- First chunk's embedding (all 384 numbers)
- Sample embeddings from multiple chunks
- Similarity between chunks

In [ ]:
# Load embedding model
print(f"🔄 Loading embedding model: {EMBEDDING_MODEL}")
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
print(f"✅ Model loaded!")
print(f"   Model Dimension: {embedding_model.get_sentence_embedding_dimension()}")

In [ ]:
# Generate embeddings for all chunks
print(f"\n🔄 Generating embeddings for {len(chunks)} chunks...")
embeddings = embedding_model.encode(chunks, show_progress_bar=True)

print(f"\n📊 Embedding Results:")
print(f"   Embeddings Shape: {embeddings.shape}")
print(f"   Number of Chunks: {embeddings.shape[0]}")
print(f"   Embedding Dimension: {embeddings.shape[1]}")
print(f"   Total Numbers: {embeddings.shape[0] * embeddings.shape[1]:,}")
print("\n✅ Embedding generation complete!")

In [ ]:
# 🔍 DEBUG: Show embedding for first chunk
print("="*80)
print("🔍 DEBUG: Embedding for Chunk 1")
print("="*80)
print(f"\n📍 Chunk 1 Text (first 200 chars):")
print(chunks[0][:200])
print("\n📍 Chunk 1 Embedding (all 384 numbers):")
print(embeddings[0])
print(f"\n📊 Statistics:")
print(f"   Min value: {embeddings[0].min():.6f}")
print(f"   Max value: {embeddings[0].max():.6f}")
print(f"   Mean value: {embeddings[0].mean():.6f}")
print(f"   Std deviation: {embeddings[0].std():.6f}")

In [ ]:
# 🔍 DEBUG: Compare embeddings of different chunks
print("="*80)
print("🔍 DEBUG: Embedding Comparison")
print("="*80)

if len(chunks) >= 3:
    # Show first 10 dimensions of first 3 chunks
    print("\n📍 First 10 dimensions of first 3 chunks:")
    df = pd.DataFrame({
        'Chunk 1': embeddings[0][:10],
        'Chunk 2': embeddings[1][:10],
        'Chunk 3': embeddings[2][:10]
    })
    df.index = [f'Dim {i}' for i in range(10)]
    print(df.to_string())
    
    # Calculate cosine similarity
    from numpy.linalg import norm
    
    def cosine_similarity(a, b):
        return np.dot(a, b) / (norm(a) * norm(b))
    
    sim_1_2 = cosine_similarity(embeddings[0], embeddings[1])
    sim_1_3 = cosine_similarity(embeddings[0], embeddings[2])
    sim_2_3 = cosine_similarity(embeddings[1], embeddings[2])
    
    print(f"\n📊 Cosine Similarity (0=different, 1=identical):")
    print(f"   Chunk 1 vs Chunk 2: {sim_1_2:.4f} (adjacent chunks)")
    print(f"   Chunk 1 vs Chunk 3: {sim_1_3:.4f}")
    print(f"   Chunk 2 vs Chunk 3: {sim_2_3:.4f} (adjacent chunks)")
    print("\n💡 Adjacent chunks should have higher similarity due to overlap!")

## Step 1.4: Vector Storage (ChromaDB)
**Goal:** Store embeddings in a vector database for fast similarity search

**What you'll see:**
- Database collection created
- Number of vectors stored
- Sample query to verify storage

In [ ]:
# Initialize ChromaDB
print("🔄 Initializing ChromaDB...")
chroma_client = chromadb.Client()

# Create collection (delete if exists)
collection_name = "rag_debug_collection"
try:
    chroma_client.delete_collection(collection_name)
    print(f"   Deleted existing collection: {collection_name}")
except:
    pass

collection = chroma_client.create_collection(collection_name)
print(f"✅ Collection created: {collection_name}")

In [ ]:
# Store embeddings in ChromaDB
print(f"\n🔄 Storing {len(chunks)} chunks in ChromaDB...")

# Prepare IDs and metadata
ids = [f"chunk_{i}" for i in range(len(chunks))]
metadatas = [{"chunk_id": i, "length": len(chunks[i])} for i in range(len(chunks))]

# Add to collection
collection.add(
    documents=chunks,
    embeddings=embeddings.tolist(),
    ids=ids,
    metadatas=metadatas
)

print(f"✅ Stored successfully!")
print(f"   Collection: {collection_name}")
print(f"   Total Vectors: {collection.count()}")

In [ ]:
# 🔍 DEBUG: Verify storage by querying
print("="*80)
print("🔍 DEBUG: Verify Storage")
print("="*80)

# Get first few items
sample_items = collection.get(ids=["chunk_0", "chunk_1", "chunk_2"])

print(f"\n📍 Retrieved {len(sample_items['ids'])} items from database:")
for i, (id, doc, meta) in enumerate(zip(sample_items['ids'], sample_items['documents'], sample_items['metadatas'])):
    print(f"\n{'-'*80}")
    print(f"ID: {id}")
    print(f"Metadata: {meta}")
    print(f"Document (first 200 chars): {doc[:200]}...")

---
# 🎉 PART 1 COMPLETE: Indexing Done!

**Summary of what we did:**
1. ✅ Extracted text from PDF
2. ✅ Split into chunks (with overlap)
3. ✅ Generated embeddings (384-dimensional vectors)
4. ✅ Stored in ChromaDB vector database

**The document is now "indexed" and ready for questions!**

---

---
---
# PART 2: QUERYING (AUGMENTATION + GENERATION)
**This happens for EVERY question you ask**

## Step 2.1: Question Embedding
**Goal:** Convert question into same embedding space as chunks

**What you'll see:**
- Question text
- Question embedding (384 numbers)
- Comparison with chunk embeddings

In [ ]:
# Define a question
# Change this to any question about the document
QUESTION = "What was the revenue in 2009?"

print("="*80)
print("❓ User Question")
print("="*80)
print(f"\n{QUESTION}\n")

In [ ]:
# Generate embedding for question
print("🔄 Generating embedding for question...")
question_embedding = embedding_model.encode([QUESTION])[0]

print(f"✅ Question embedded!")
print(f"   Embedding shape: {question_embedding.shape}")
print(f"   Dimension: {len(question_embedding)}")

In [ ]:
# 🔍 DEBUG: Show question embedding
print("="*80)
print("🔍 DEBUG: Question Embedding")
print("="*80)
print(f"\n📍 Question: {QUESTION}")
print(f"\n📍 Question Embedding (all 384 numbers):")
print(question_embedding)
print(f"\n📊 Statistics:")
print(f"   Min value: {question_embedding.min():.6f}")
print(f"   Max value: {question_embedding.max():.6f}")
print(f"   Mean value: {question_embedding.mean():.6f}")
print(f"   Std deviation: {question_embedding.std():.6f}")

In [ ]:
# 🔍 DEBUG: Compare question embedding with first 3 chunk embeddings
print("="*80)
print("🔍 DEBUG: Question vs Chunk Embeddings")
print("="*80)

print("\n📍 First 10 dimensions comparison:")
df = pd.DataFrame({
    'Question': question_embedding[:10],
    'Chunk 1': embeddings[0][:10],
    'Chunk 2': embeddings[1][:10],
    'Chunk 3': embeddings[2][:10]
})
df.index = [f'Dim {i}' for i in range(10)]
print(df.to_string())

# Calculate similarity
sim_q_1 = cosine_similarity(question_embedding, embeddings[0])
sim_q_2 = cosine_similarity(question_embedding, embeddings[1])
sim_q_3 = cosine_similarity(question_embedding, embeddings[2])

print(f"\n📊 Cosine Similarity with Question:")
print(f"   Question vs Chunk 1: {sim_q_1:.4f}")
print(f"   Question vs Chunk 2: {sim_q_2:.4f}")
print(f"   Question vs Chunk 3: {sim_q_3:.4f}")
print("\n💡 Higher similarity = more relevant to the question!")

## Step 2.2: Semantic Search (RETRIEVAL)
**Goal:** Find most similar chunks to the question

**What you'll see:**
- TOP_K most relevant chunks
- Similarity scores for each
- Full text of retrieved chunks
- Why each chunk was retrieved

In [ ]:
# Search ChromaDB for similar chunks
print(f"🔍 Searching for TOP {TOP_K} most similar chunks...")

results = collection.query(
    query_embeddings=[question_embedding.tolist()],
    n_results=TOP_K
)

retrieved_chunks = results['documents'][0]
retrieved_ids = results['ids'][0]
retrieved_distances = results['distances'][0]
retrieved_metadatas = results['metadatas'][0]

print(f"✅ Retrieved {len(retrieved_chunks)} chunks!")
print(f"   Chunk IDs: {retrieved_ids}")

In [ ]:
# 🔍 DEBUG: Show all retrieved chunks with scores
print("="*80)
print("🔍 DEBUG: Retrieved Chunks (RETRIEVAL Results)")
print("="*80)
print(f"\n❓ Question: {QUESTION}")

for i, (chunk_id, chunk_text, distance, metadata) in enumerate(zip(
    retrieved_ids, retrieved_chunks, retrieved_distances, retrieved_metadatas
)):
    # Convert distance to similarity (ChromaDB uses L2 distance)
    similarity = 1 / (1 + distance)
    
    print(f"\n{'='*80}")
    print(f"RETRIEVED CHUNK {i+1}/{TOP_K}")
    print(f"{'='*80}")
    print(f"Chunk ID: {chunk_id}")
    print(f"Distance: {distance:.4f} (lower = more similar)")
    print(f"Similarity Score: {similarity:.4f} (higher = more similar)")
    print(f"Metadata: {metadata}")
    print(f"\nFull Chunk Text:")
    print(f"{'-'*80}")
    print(chunk_text)
    print(f"{'-'*80}")

In [ ]:
# 🔍 DEBUG: Why were these chunks retrieved?
print("="*80)
print("🔍 DEBUG: Why These Chunks Were Retrieved")
print("="*80)

print(f"\n❓ Question: {QUESTION}")
print(f"\n🔑 Key words in question: {set(QUESTION.lower().split())}")

for i, chunk in enumerate(retrieved_chunks):
    print(f"\n{'-'*80}")
    print(f"Chunk {i+1}:")
    
    # Find matching words
    question_words = set(QUESTION.lower().split())
    chunk_words = set(chunk.lower().split())
    matching_words = question_words.intersection(chunk_words)
    
    print(f"   Matching words: {matching_words if matching_words else 'None (semantic similarity)'}")
    
    # Show context around matching words
    if matching_words:
        for word in matching_words:
            if len(word) > 2:  # Skip short words
                idx = chunk.lower().find(word)
                if idx != -1:
                    start = max(0, idx - 50)
                    end = min(len(chunk), idx + len(word) + 50)
                    context = chunk[start:end]
                    print(f"   Context for '{word}': ...{context}...")
                    break

## Step 2.3: Context Building (AUGMENTATION)
**Goal:** Combine retrieved chunks into context for LLM

**What you'll see:**
- Combined context from all chunks
- Total context length
- How chunks are arranged

In [ ]:
# Build context from retrieved chunks
print("🔄 Building context from retrieved chunks...")

context = "\n\n".join([
    f"[Chunk {i+1}]\n{chunk}"
    for i, chunk in enumerate(retrieved_chunks)
])

print(f"\n📊 Context Statistics:")
print(f"   Number of chunks combined: {len(retrieved_chunks)}")
print(f"   Total context length: {len(context):,} characters")
print(f"   Total context words: {len(context.split()):,} words")
print(f"   Average chunk length: {len(context) / len(retrieved_chunks):.0f} characters")
print("\n✅ Context built!")

In [ ]:
# 🔍 DEBUG: Show full combined context
print("="*80)
print("🔍 DEBUG: Full Combined Context (AUGMENTATION)")
print("="*80)
print(f"\nThis is what the LLM will see:")
print("="*80)
print(context)
print("="*80)

## Step 2.4: Prompt Building
**Goal:** Create prompt with context + question

**What you'll see:**
- Complete prompt sent to LLM
- Prompt structure
- Prompt length

In [ ]:
# Build prompt
prompt = f"""Based on the following context from a document, answer the question.

CONTEXT:
{context}

QUESTION:
{QUESTION}

ANSWER:"""

print(f"📊 Prompt Statistics:")
print(f"   Prompt length: {len(prompt):,} characters")
print(f"   Prompt words: {len(prompt.split()):,} words")
print(f"   Estimated tokens: {len(prompt.split()) * 1.3:.0f} tokens")
print("\n✅ Prompt built!")

In [ ]:
# 🔍 DEBUG: Show complete prompt
print("="*80)
print("🔍 DEBUG: Complete Prompt Sent to LLM")
print("="*80)
print("\nThis is EXACTLY what goes to the LLM:")
print("="*80)
print(prompt)
print("="*80)

## Step 2.5: LLM Answer Generation
**Goal:** Get answer from LLM

**Note:** This step requires Together AI API key.
For this debug notebook, we'll simulate what would happen.

**What you would see:**
- API call to Nemotron
- Generated answer
- Response metadata

In [ ]:
# SIMULATION: What the LLM API call looks like
print("="*80)
print("🔍 DEBUG: LLM API Call (SIMULATED)")
print("="*80)

print("\n📡 API Request:")
print("-"*80)
api_request = {
    "model": "nvidia/nemotron-3-ultra-550b-a55b",
    "prompt": prompt[:200] + "... [truncated]",  # Show first 200 chars
    "max_tokens": 500,
    "temperature": 0.0,
    "stop": ["QUESTION:", "\n\n"]
}
pprint(api_request)

print("\n" + "="*80)
print("⚠️ ACTUAL LLM CALL SKIPPED (requires API key)")
print("="*80)
print("\n📝 To enable LLM calls:")
print("   1. Install: pip install together")
print("   2. Set API key: export TOGETHER_API_KEY='your_key'")
print("   3. Uncomment code in next cell")
print("\n💡 For now, we'll show what the response would look like...")

In [ ]:
# OPTIONAL: Uncomment to make actual LLM call
# import together
# together.api_key = "your_api_key_here"
# 
# response = together.Complete.create(
#     model="nvidia/nemotron-3-ultra-550b-a55b",
#     prompt=prompt,
#     max_tokens=500,
#     temperature=0.0
# )
# 
# answer = response['output']['choices'][0]['text'].strip()
# print(f"\n🤖 LLM Answer:\n{answer}")

# SIMULATED ANSWER
simulated_answer = "[This is where the LLM's answer would appear]\n\nExample: Based on the context, the revenue in 2009 was $2.4 billion."

print("="*80)
print("🔍 DEBUG: LLM Response (SIMULATED)")
print("="*80)
print(f"\n🤖 Generated Answer:")
print("-"*80)
print(simulated_answer)
print("-"*80)

---
# 🎉 PART 2 COMPLETE: Query Pipeline Done!

**Summary of what we did:**
1. ✅ Embedded the question (384-dimensional vector)
2. ✅ Searched ChromaDB for similar chunks (RETRIEVAL)
3. ✅ Combined retrieved chunks into context (AUGMENTATION)
4. ✅ Built prompt with context + question
5. ✅ (Would) Send to LLM for answer (GENERATION)

**The complete RAG pipeline is now traced!**

---

---
---
# 📊 SUMMARY: Complete RAG Pipeline

## Part 1: INDEXING (One-time)
```
PDF File
   ↓
Text Extraction (PyPDF2)
   ↓
76,000 words
   ↓
Text Chunking (LangChain)
   ↓
150 chunks × 1500 chars
   ↓
Embedding (all-MiniLM-L6-v2)
   ↓
150 vectors × 384 dimensions
   ↓
ChromaDB Storage
```

## Part 2: QUERYING (Every question)
```
Question: "What was revenue in 2009?"
   ↓
Question Embedding
   ↓
1 vector × 384 dimensions
   ↓
Semantic Search (ChromaDB)
   ↓
TOP 5 most similar chunks
   ↓
Context Building
   ↓
Combined text: ~7,500 characters
   ↓
Prompt = Context + Question
   ↓
LLM (Nemotron)
   ↓
Answer: "Revenue was $2.4B"
```

---

# 🎯 Key Insights from Debug Logs

## What We Learned:

### 1. PDF Extraction
- Raw text is messy (formatting issues, tables mangled)
- Not all text is equally important
- Page boundaries are artificial

### 2. Chunking
- Chunks overlap to prevent losing context
- Chunk size matters (too small = incomplete context, too large = diluted relevance)
- Adjacent chunks have high similarity due to overlap

### 3. Embeddings
- 384 numbers represent semantic meaning
- Similar text → similar numbers
- Embeddings capture meaning, not just keywords

### 4. Retrieval
- Semantic search finds relevant chunks even without exact keyword matches
- TOP_K determines how much context LLM sees
- Retrieved chunks might not contain the answer (retrieval failure)

### 5. Context Building
- Multiple chunks combined = more complete picture
- Order matters (most relevant first)
- Total context must fit in LLM's window

### 6. Generation
- LLM only sees retrieved chunks, not full document
- Answer quality depends on retrieval quality
- If answer not in context, LLM can't answer

---

# 🔬 Experiment: Try Different Questions

Go back to **Step 2.1** and change the `QUESTION` variable to try:
- Easy questions (single fact)
- Hard questions (multi-step reasoning)
- Questions not in document

Observe how different questions retrieve different chunks!

---

# 📚 Next Steps

1. **Try different PDFs** - Change `PDF_PATH` in Step 1.1
2. **Adjust hyperparameters:**
   - `CHUNK_SIZE` (500, 1000, 1500, 2000)
   - `CHUNK_OVERLAP` (50, 100, 200)
   - `TOP_K` (3, 5, 10, 15)
3. **Try different embedding models:**
   - `all-MiniLM-L6-v2` (current)
   - `all-mpnet-base-v2` (better quality, slower)
   - `paraphrase-MiniLM-L6-v2` (for paraphrased queries)
4. **Compare with your Sprint 3 results!**

---

**This notebook shows EXACTLY what happens in your RAG pipeline!** 🎉